# Spaceship Titanic: Пошаговое решение ML задачи

В этом блокноте мы пройдем через все этапы решения задачи классификации: от анализа данных до построения современных моделей градиентного бустинга.

## 0. Исправление бинарной несовместимости (Fix ValueError: numpy.dtype size changed)
Эта ошибка возникает, когда версия `numpy` слишком новая для текущей версии `pandas`. Мы принудительно установим стабильную версию `numpy 1.26.4`.

In [ ]:
import sys
!{sys.executable} -m pip install "numpy<2.0" "pandas>=2.0" catboost shap scikit-learn matplotlib seaborn

## 1. Загрузка библиотек и данных

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import shap

import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Размер обучающей выборки: {train_df.shape}")
print(f"Размер тестовой выборки: {test_df.shape}")

Размер обучающей выборки: (8693, 14)
Размер тестовой выборки: (4277, 13)


## 2. Исследовательский анализ данных (EDA)

### Зачем мы это делаем?
Анализ данных — это не просто построение красивых графиков. На этом этапе мы ищем **закономерности**, которые лягут в основу модели. 

**Почему мы начинаем именно так и что мы делаем:**
1. **Проверка баланса классов**: Мы изучаем распределение целевой переменной `Transported`. Если бы классы были сильно смещены (например, 99% на 1%), стандартные метрики и алгоритмы работали бы плохо. В нашем случае баланс почти идеальный (50/50), что позволяет использовать Accuracy.
2. **Поиск ключевых драйверов (Корреляций)**: Мы изучаем, как каждый признак влияет на результат. Например, мы обнаружили, что `CryoSleep` (Криосон) — это «золотой» признак. Пассажиры в криосне имеют аномально высокий шанс перемещения. 

**Почему нельзя по-другому или почему будет хуже:**
* **Проблема слепого обучения**: Если мы не проведем EDA и просто «скормим» данные модели, мы не узнаем о скрытых связях. Например, мы не поймем, что `Cabin` — это не просто текст, а комбинация палубы и стороны. Без EDA мы бы оставили этот признак как есть, и модель не смогла бы извлечь из него пользу.
* **Риск неправильной валидации**: Если не увидеть, что данные распределены равномерно, можно выбрать неправильную стратегию тестирования. Без EDA мы могли бы получить «красивые» цифры на обучении, которые бы полностью провалились на реальном тесте Kaggle.

--- 

### 💡 Подсказка: Стратегии борьбы с пропусками
Когда мы видим пропуски в данных, у нас есть несколько путей:
1. **Удаление**: Самый простой, но рискованный путь (теряем данные). В нашей задаче это лишит нас 25% информации.
2. **Простое заполнение**: Средним или медианой. Быстро, но игнорирует контекст. Если заполнить траты медианой для всех, мы «сломаем» логику криосна.
3. **Логическое заполнение (наш выбор)**: Например, если пассажир в криосне (`CryoSleep=True`), его траты гарантированно равны 0. Или если мы знаем планету одного члена семьи, мы можем присвоить её остальным.
4. **Алгоритмическое заполнение**: Использование моделей (KNN, IterativeImputer) для предсказания пропущенных значений.
5. **Автоматическая обработка**: Градиентный бустинг (CatBoost/XGBoost) умеет работать с пропусками сам.

**Почему мы не используем только медиану?**
Потому что это создаст «шум». Обучение на логически выверенных данных (например, восстановление планеты по группе) дает прирост в 1-2% точности, что критично для соревнований.

In [ ]:
plt.figure(figsize=(15, 5))

# Целевая переменная
plt.subplot(1, 3, 1)
sns.countplot(data=train_df, x='Transported')
plt.title('Распределение Transported')

# Влияние криосна
plt.subplot(1, 3, 2)
sns.countplot(data=train_df, x='CryoSleep', hue='Transported')
plt.title('CryoSleep vs Transported')

# Влияние планеты
plt.subplot(1, 3, 3)
sns.countplot(data=train_df, x='HomePlanet', hue='Transported')
plt.title('HomePlanet vs Transported')

plt.show()

## 3. Предобработка данных и Feature Engineering

Здесь мы извлекаем новые признаки из `PassengerId` и `Cabin`, а также заполняем пропуски.

### ❓ А можно ли просто удалить строки с пропусками?
Это популярный вопрос, но в данной задаче удаление — плохая стратегия:
1. **Огромные потери**: Пропуски распределены так, что удалив все строки с NaN, мы потеряем **~25% данных** (более 2000 строк). Модель станет слабее.
2. **Проблема Теста**: В тестовом наборе (`test.csv`) тоже есть пропуски, но мы **обязаны** дать предсказание для каждого пассажира. Мы не можем просто «удалить» пассажира из теста. Значит, логика заполнения нам всё равно нужна.
3. **Умные связи**: Многие пропуски легко восстановимы по группе или палубе. Удаление таких строк — это выбрасывание полезной информации.

--- 

### ⚠️ Риск внесения шума при финальном заполнении
Может возникнуть вопрос: не слишком ли много шума мы вносим, заполняя остатки данных медианой или модой?
*   **Минимальный объем**: После «умного» заполнения (по группам) количество пропусков падает до **1-3%**. Шум от такого количества данных пренебрежимо мал.
*   **Баланс Шум vs Сигнал**: Заполняя медианой одну колонку (например, Возраст), мы сохраняем все остальные колонки для этого пассажира. **Сигнал** от всех остальных признаков гораздо сильнее, чем **шум** от одного среднего значения.
*   **Устойчивость моделей**: Градиентный бустинг обучается на тысячах деревьев. Ошибка в одном «шумном» значении медианы легко нивелируется другими деревьями.

In [ ]:
def get_missing_stats(df, label):
    rows_with_nan = df.isnull().any(axis=1).sum()
    pct = (rows_with_nan / len(df)) * 100
    print(f"{label}: {rows_with_nan} человек ({pct:.2f}%) имеют хотя бы один пропуск")

def preprocess_with_stats(df_input):
    df = df_input.copy()
    print("--- Анализ пропусков ---")
    get_missing_stats(df, "До обработки")
    
    # 1. Извлечение группы и первичная логика
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])
    df['GroupSize'] = df.groupby('Group')['Group'].transform('count')
    
    # 2. Умное заполнение (Групповое)
    df['HomePlanet'] = df.groupby('Group')['HomePlanet'].transform(lambda x: x.ffill().bfill())
    
    # 3. Умное заполнение (Криосон и Траты)
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in spend_cols:
        df[col] = df[col].fillna(0) # Если не указано, считаем 0
    df['TotalSpending'] = df[spend_cols].sum(axis=1)
    
    df.loc[(df['CryoSleep'].isnull()) & (df['TotalSpending'] > 0), 'CryoSleep'] = False
    df.loc[(df['CryoSleep'].isnull()) & (df['TotalSpending'] == 0), 'CryoSleep'] = True
    
    get_missing_stats(df, "После 'умного' заполнения (группы + логика трат)")
    
    # 4. Финальное заполнение (Медианы и Моды)
    df['HomePlanet'] = df['HomePlanet'].fillna(df['HomePlanet'].mode()[0])
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['VIP'] = df['VIP'].fillna(False).astype(int)
    df['Cabin'] = df['Cabin'].fillna('Unknown/Unknown/Unknown')
    df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['CryoSleep'] = df['CryoSleep'].astype(bool)
    
    get_missing_stats(df, "После полного заполнения всех данных")
    print("-----------------------\n")
    
    # 5. Кодирование
    cat_cols = ['HomePlanet', 'Deck', 'Side']
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    
    features = ['Age', 'TotalSpending', 'GroupSize', 'CryoSleep', 'VIP'] + \
               [col for col in df.columns if col.startswith(('HomePlanet_', 'Deck_', 'Side_'))] + spend_cols
    
    return df[features]

X_train = preprocess_with_stats(train_df)
y_train = train_df['Transported'].astype(int)
X_test = preprocess_with_stats(test_df)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Методология оценки и выбор метрик

Для оценки моделей мы используем **Stratified K-Fold Cross-Validation** и набор метрик. Давайте разберем, почему именно их:

### Почему Stratified K-Fold?
1. **Стабильность**: Вместо одного разбиения на Train/Test мы делим данные на 5 частей (фолдов). Модель обучается 5 раз, каждый раз используя новый фолд для теста. Это дает нам среднюю точность и понимание разброса (дисперсии).
2. **Стратификация**: Гарантирует, что в каждом фолде пропорция классов (Transported True/False) такая же, как в исходном наборе. Это критично для надежной оценки.

### Обоснование метрик:
*   **Accuracy (Точность)**: Главная метрика соревнования Kaggle. Показывает процент правильных предсказаний.
*   **Precision (Точность)**: Какая доля из тех, кого мы назвали «перемещенными», действительно переместилась. Важна, если цена «ложного срабатывания» высока.
*   **Recall (Полнота)**: Какую долю реально перемещенных пассажиров мы смогли найти. Важна, чтобы не пропустить никого.
*   **F1-Score**: Гармоническое среднее между Precision и Recall. Лучшая метрика, если мы хотим сбалансировать точность и полноту.
*   **ROC-AUC**: Показывает способность модели разделять классы независимо от выбранного порога вероятности. Чем выше AUC, тем лучше модель отличает «перемещенных» от «оставшихся».

### Почему не другие тесты?
*   **Log-Loss**: Хорошая метрика для вероятностей, но она очень сильно штрафует за «уверенные ошибки». Так как Kaggle оценивает только по итоговому флагу (True/False), Accuracy здесь более репрезентативна для лидерборда.
*   **Статистические тесты (p-value)**: Они полезны для проверки гипотез (например, «действительно ли выживаемость на палубе B выше?»), но в задачах предсказания мы фокусируемся на прогностической силе модели, а не на статистической значимости отдельных коэффициентов.

## 5. Сравнение моделей и глубокий разбор параметров

Здесь мы переходим к самым мощным ансамблевым алгоритмам. Они работают по разным принципам, и их параметры критически важны для успеха.

### 🌲 Случайный лес (Random Forest)
Это метод **бэггинга** (Bagging). Он строит много независимых деревьев и усредняет их ответы.
*   **n_estimators=100**: Количество деревьев. Чем их больше, тем стабильнее модель, но дольше обучение. Обычно 100-500 достаточно.
*   **n_jobs=-1**: Параметр параллелизма. Модель задействует **все ядра вашего процессора**, строя деревья одновременно. Это дает колоссальный прирост в скорости.
*   **max_features**: Сколько случайных признаков выбирать для каждого разбиения. Это «секретный ингредиент», который делает деревья разными и предотвращает их одинаковость.

### 🚀 Градиентный бустинг (CatBoost)
Это метод **бустинга** (Boosting). В отличие от леса, деревья здесь строятся **последовательно**. Каждое новое дерево учится на ошибках предыдущих.
*   **iterations=500**: Количество шагов (деревьев). Слишком много итераций может привести к переобучению, когда модель просто «зазубрит» данные.
*   **learning_rate=0.05**: Скорость обучения. Определяет, насколько сильно мы доверяем каждому новому дереву. Маленький шаг (0.01-0.1) требует больше итераций, но дает более точный результат.
*   **depth=6**: Глубина деревьев. Бустинг лучше работает с неглубокими деревьями (обычно от 4 до 8). Глубина 6 позволяет уловить сложные взаимодействия признаков (например, Влияние Возраста + Планеты вместе).
*   **verbose=0**: Убирает вывод технической информации для чистоты блокнота.

### 🧪 Линейные модели и Одиночные деревья
*   **Logistic Regression (C=1.0)**: Мы используем ее как базовый уровень (Baseline). Параметр `C` контролирует регуляризацию (борьбу со слишком сложными весами).
*   **Decision Tree (max_depth=5)**: Простейшее дерево для визуализации логики.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(C=1.0),
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1),
    "CatBoost": CatBoostClassifier(iterations=500, verbose=0, learning_rate=0.05, depth=6)
}

results = {}
for name, model in models.items():
    cv = cross_validate(model, X_train_scaled if "Logistic" in name else X_train, y_train, cv=5, scoring='accuracy')
    results[name] = cv['test_score'].mean()
    print(f"{name}: {results[name]:.4f}")

## 6. Глубокий анализ лучшей модели (CatBoost + SHAP)

### Что такое SHAP?
**SHAP (Shapley Additive Explanations)** — это метод из теории игр, который позволяет «заглянуть в мозг» сложной модели. Он показывает, какой вклад внес каждый конкретный признак в финальное предсказание.

**Как читать график Summary Plot:**
1. **Важность (сверху вниз)**: Самые важные признаки находятся на самом верху. 
2. **Цвет точки**: 
   * **Красный** — высокое значение признака (например, большой возраст или наличие криосна).
   * **Синий** — низкое значение признака.
3. **Положение на оси X**: 
   * Если точки уходят **вправо** — этот признак «толкает» прогноз к значению **True** (перемещен).
   * Если точки уходят **влево** — признак «толкает» прогноз к значению **False** (остался).

**Пример**: Если мы видим красные точки у `CryoSleep` далеко справа, это значит: «Наличие криосна сильно повышает шансы на перемещение».

In [ ]:
best_model = CatBoostClassifier(iterations=1000, verbose=0).fit(X_train, y_train)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_train)

plt.title('SHAP: Влияние признаков')
shap.summary_plot(shap_values, X_train)

## 8. Эксперимент: Регрессия как классификатор

А что, если мы попробуем решить эту задачу не классификацией, а регрессией? 

**Логика эксперимента**:
1. Превращаем целевую переменную в числа: `False -> -1`, `True -> 1`.
2. Обучаем линейную регрессию предсказывать это число.
3. Применяем правило: Если результат `< 0` -> это `False`, если `>= 0` -> это `True`.

**Зачем это нужно?**
Иногда регрессия лучше улавливает «степень уверенности» и дает более гладкую границу разделения. Мы попробуем обычную регрессию и ее варианты с регуляризацией (**Ridge** и **Lasso**).

In [19]:
# Подготовка данных для регрессии
y_reg = y_train.map({0: -1, 1: 1})

reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01)
}

print("--- Результаты регрессионного подхода ---")
for name, model in reg_models.items():
    # Кросс-валидация для регрессии
    cv_results = cross_validate(model, X_train_scaled, y_reg, cv=5)
    
    # Для расчета Accuracy нам нужно сделать предсказания
    model.fit(X_train_scaled, y_reg)
    y_pred_cont = model.predict(X_train_scaled)
    y_pred_class = (y_pred_cont >= 0).astype(int)
    y_train_class = (y_reg >= 0).astype(int)
    
    acc = accuracy_score(y_train_class, y_pred_class)
    print(f"{name}: Accuracy = {acc:.4f}")

--- Результаты регрессионного подхода ---
Linear Regression: Accuracy = 0.7697
Ridge Regression: Accuracy = 0.7698
Lasso Regression: Accuracy = 0.7688


## 9. Финальное предсказание

In [20]:
predictions = best_model.predict(X_test)
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': predictions.astype(bool)
})
submission.to_csv('final_submission.csv', index=False)
print("Предсказания сохранены в final_submission.csv")

Предсказания сохранены в final_submission.csv
